# 04 — VADER Sentiment Analysis

**VADER** is a rule-based NLP tool with 7,500+ pre-scored lexicon entries. For war journalism, it captures raw vocabulary intensity — the same signal that commodity traders respond to.

| Section | Content |
|---|---|
| 1 | Apply VADER to all 314 articles |
| 2 | Sentiment distribution |
| 3 | Word clouds by sentiment class |
| 4 | Compound score statistics |
| 5 | Daily aggregation and time trend |

> **Next step:** `05_RoBERTa_Sentiment_Analysis.ipynb`

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib
matplotlib.use('Agg')
from src.config import NEWS_PROCESSED_FILE

proc_df = pd.read_csv(NEWS_PROCESSED_FILE)
print(f'Articles to score: {len(proc_df)}')

## 1. Apply VADER

Compound score thresholds: >= 0.05 = Positive, <= -0.05 = Negative, else Neutral.

In [ ]:
from src.sentiment import apply_vader

df_vader = apply_vader(proc_df)
print('VADER columns added:', [c for c in df_vader.columns if 'vader' in c])
df_vader[['title', 'vader_compound', 'vader_label']].head(10)

## 2. Sentiment Distribution

In [ ]:
counts = df_vader['vader_label'].value_counts()
total  = len(df_vader)
print('VADER Sentiment Distribution:')
for label, n in counts.items():
    print(f'  {label:10s}: {n:4d}  ({100*n/total:.1f}%)')
print(f'\nMean compound score: {df_vader["vader_compound"].mean():.4f}')

In [ ]:
from src.visualize import plot_sentiment_distribution

plot_sentiment_distribution(df_vader)
print('Saved: images/07_sentiment_distribution.png')

## 3. Word Clouds by Sentiment Class

What vocabulary drives each sentiment bucket?

In [ ]:
from src.visualize import plot_wordcloud

plot_wordcloud(df_vader, label_col='vader_label')
print('Saved: images/05_wordcloud_by_sentiment.png')

neg_text   = df_vader[df_vader['vader_label'] == 'Negative']['cleaned_text']
neg_tokens = neg_text.str.split().explode()
print('\nTop 15 Negative tokens:')
print(neg_tokens.value_counts().head(15).to_string())

## 4. Compound Score Statistics by Label

In [ ]:
print('Compound score breakdown by label:')
print(df_vader.groupby('vader_label')['vader_compound'].describe().round(4).to_string())

print('\nSample high-confidence Negative articles:')
neg_sample = df_vader.nsmallest(3, 'vader_compound')[['title', 'vader_compound']]
for _, r in neg_sample.iterrows():
    print(f'  [{r["vader_compound"]:.3f}] {r["title"][:90]}')

## 5. Daily Aggregation and Sentiment Over Time

In [ ]:
from src.sentiment import aggregate_daily
from src.visualize import plot_sentiment_over_time

daily_df = aggregate_daily(df_vader)
plot_sentiment_over_time(daily_df)
print('Saved: images/08_sentiment_over_time.png')
print(f'\nDaily sentiment: {len(daily_df)} days')
daily_df.head(10)

## Key Findings

| Metric | Value |
|---|---|
| Negative articles | 191 (60.8%) |
| Neutral articles | 65 (20.7%) |
| Positive articles | 58 (18.5%) |
| Key negative tokens | attack, sanction, threat, conflict |

War journalism uses consistently negative vocabulary. VADER captures this intensity directly.

> **Next step:** `05_RoBERTa_Sentiment_Analysis.ipynb`